# CLIP K-Fold Training from Manifests

This notebook mirrors the ResNet K-fold workflow, but replaces end-to-end CNN fine-tuning with:

1. CLIP image embedding extraction
2. Stratified K-Fold training of a lightweight classifier on embeddings
3. OOF threshold tuning
4. Best-fold and fold-ensemble submissions

Outputs are saved with the same naming style as the ResNet notebook:
- `oof_predictions.csv`
- `test_probabilities.csv`
- `submission_best_fold.csv`
- `submission_fold_ensemble.csv`
- `metrics_summary.json`
- `config.json`

In [42]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from IPython.display import display

from pandas.api.types import is_numeric_dtype
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold

from transformers import CLIPModel, CLIPProcessor

## 1. Configuration

In [43]:
CONFIG = {
    # --- project / files ---
    "project_root_override": None,
    "root_markers": ["data2", "outputs"],

    # Choose the training manifest to use.
    # Recommended order for experiments:
    # 1) train_manifest_etsy_only.csv
    # 2) train_manifest_etsy_plus_external.csv
    # 3) later: a balanced external version if you create one
    "train_manifest_relpath": "outputs/manifests/train_manifest_etsy_only.csv",
    "test_manifest_relpath": "outputs/manifests/test_manifest_etsy.csv",

    # Experiment name controls output folders/files
    "experiment_name": "clip_logreg_etsy_only_v1",

    # --- labels ---
    "positive_label": "ai_generated",
    "negative_label": "authentic",

    # --- CLIP / embeddings ---
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "clip_model_name": "openai/clip-vit-base-patch32",
    "batch_size": 32,                # embedding extraction batch size
    "normalize_embeddings": True,
    "cache_embeddings": True,

    # --- CV / classifier ---
    "random_state": 42,
    "n_splits": 5,
    "classifier_type": "logreg",     # currently: "logreg"
    "logreg_C": 1.0,
    "logreg_max_iter": 5000,

    # --- imbalance handling ---
    "use_class_weights": True,

    # --- misc ---
    "save_oof_csv": True,
    "save_test_probs_csv": True,
    "save_submission_csv": True,
}

In [63]:
def find_project_root(start: Path, markers: list[str]) -> Path:
    start = start.resolve()
    current = start
    while True:
        if all((current / marker).exists() for marker in markers):
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find project root from {start}. Expected markers: {markers}"
            )
        current = current.parent


def _resolve_path(project_root: Path, path_like: str | Path) -> Path:
    p = Path(path_like).expanduser()
    return p if p.is_absolute() else (project_root / p)


def load_rgb_image(path: str | Path) -> Image.Image:
    try:
        return Image.open(path).convert("RGB")
    except (UnidentifiedImageError, OSError, FileNotFoundError) as e:
        raise RuntimeError(f"Failed to open image: {path}") from e


cwd = Path.cwd()

if CONFIG["project_root_override"]:
    PROJECT_ROOT = Path(CONFIG["project_root_override"]).expanduser().resolve()
else:
    PROJECT_ROOT = find_project_root(cwd, CONFIG["root_markers"])

TRAIN_MANIFEST_PATH = _resolve_path(PROJECT_ROOT, CONFIG["train_manifest_relpath"])
TEST_MANIFEST_PATH = _resolve_path(PROJECT_ROOT, CONFIG["test_manifest_relpath"])


OUTPUT_DIR = PROJECT_ROOT / "outputs" / CONFIG["experiment_name"]

EXPERIMENT_DIR = PROJECT_ROOT / "outputs" / "experiments" / CONFIG["experiment_name"]
MODELS_DIR = EXPERIMENT_DIR / "models"
REPORTS_DIR = EXPERIMENT_DIR / "reports"
SUBMISSIONS_DIR = EXPERIMENT_DIR / "submissions"
CACHE_DIR = EXPERIMENT_DIR / "cache"







for d in [OUTPUTS_DIR, EXPERIMENT_DIR, MODELS_DIR, CACHE_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with open(REPORTS_DIR / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=4)

print("PROJECT_ROOT       :", PROJECT_ROOT)
print("TRAIN_MANIFEST_PATH:", TRAIN_MANIFEST_PATH)
print("TEST_MANIFEST_PATH :", TEST_MANIFEST_PATH)
print("EXPERIMENT_DIR     :", EXPERIMENT_DIR)
print("MODELS_DIR         :", MODELS_DIR)
print("CACHE_DIR          :", CACHE_DIR)
print("REPORTS_DIR        :", REPORTS_DIR)
print("DEVICE             :", CONFIG["device"])

PROJECT_ROOT       : /home/tbeliere/Projects/DCU/AML_PJ
TRAIN_MANIFEST_PATH: /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_etsy_only.csv
TEST_MANIFEST_PATH : /home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/test_manifest_etsy.csv
EXPERIMENT_DIR     : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1
MODELS_DIR         : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1/models
CACHE_DIR          : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1/cache
REPORTS_DIR        : /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1/reports
DEVICE             : cuda


## 2. Load manifests

In [45]:
train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
test_df = pd.read_csv(TEST_MANIFEST_PATH)

required_train_cols = {"image_id", "image_path", "label"}
required_test_cols = {"image_id", "image_path"}

missing_train = required_train_cols - set(train_df.columns)
missing_test = required_test_cols - set(test_df.columns)

if missing_train:
    raise ValueError(f"Train manifest missing columns: {missing_train}")
if missing_test:
    raise ValueError(f"Test manifest missing columns: {missing_test}")

train_df["image_path"] = train_df["image_path"].astype(str)
test_df["image_path"] = test_df["image_path"].astype(str)

if is_numeric_dtype(train_df["label"]):
    train_df["target"] = train_df["label"].astype(int)
else:
    label2id = {
        CONFIG["negative_label"]: 0,
        CONFIG["positive_label"]: 1,
    }
    train_df["target"] = train_df["label"].astype(str).str.strip().map(label2id)

    if train_df["target"].isna().any():
        bad_labels = sorted(train_df.loc[train_df["target"].isna(), "label"].astype(str).unique())
        raise ValueError(f"Unmapped labels found in train_df['label']: {bad_labels}")

    train_df["target"] = train_df["target"].astype(int)

id2label = {
    0: CONFIG["negative_label"],
    1: CONFIG["positive_label"],
}

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("\nTrain label distribution:")
display(train_df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))
display(train_df.head())
display(test_df.head())

Train shape: (4800, 13)
Test shape : (2058, 12)

Train label distribution:


,label,count
0,authentic,2485
1,ai_generated,2315


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio,target
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000,1
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000,1
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,1024,RGB,JPEG,1.000000,1
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,896,1195,RGB,JPEG,0.749791,1
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,ai_generated,etsy,train,True,True,1024,558,RGB,JPEG,1.835125,1


,image_id,image_path,label,source,split,file_exists,valid_image,width,height,mode,format,aspect_ratio
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1195,896,RGB,JPEG,1.333705
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,1024,1024,RGB,JPEG,1.000000
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,/home/tbeliere/Projects/DCU/AML_PJ/data2/etsy_...,NaN,etsy,test,True,True,896,1195,RGB,JPEG,0.749791


In [46]:
n_missing_train = (~train_df["image_path"].apply(lambda p: Path(p).exists())).sum()
n_missing_test = (~test_df["image_path"].apply(lambda p: Path(p).exists())).sum()

print("Missing train image files:", int(n_missing_train))
print("Missing test image files :", int(n_missing_test))

if n_missing_train > 0 or n_missing_test > 0:
    raise FileNotFoundError("One or more image paths in the manifests do not exist.")

Missing train image files: 0
Missing test image files : 0


## 3. CLIP loading and embedding extraction

In [47]:
DEVICE = torch.device(CONFIG["device"])
PROCESSOR = CLIPProcessor.from_pretrained(CONFIG["clip_model_name"])
CLIP = CLIPModel.from_pretrained(CONFIG["clip_model_name"]).to(DEVICE)
CLIP.eval()

print("Loaded:", CONFIG["clip_model_name"])

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5940.64it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: openai/clip-vit-base-patch32


In [48]:
@torch.no_grad()
def extract_clip_embeddings(
    image_paths,
    processor,
    model,
    device,
    batch_size=32,
    normalize_embeddings=True,
):
    all_embeddings = []

    for start in tqdm(range(0, len(image_paths), batch_size), desc="Extracting CLIP embeddings"):
        batch_paths = image_paths[start:start + batch_size]
        images = [Image.open(p).convert("RGB") for p in batch_paths]

        inputs = processor(images=images, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model.get_image_features(**inputs)

        # In normal CLIP usage this should already be a tensor.
        # But this guards against model wrappers returning an output object.
        if hasattr(outputs, "pooler_output"):
            features = outputs.pooler_output
        elif hasattr(outputs, "last_hidden_state"):
            features = outputs.last_hidden_state[:, 0]
        else:
            features = outputs

        if normalize_embeddings:
            features = features / features.norm(dim=-1, keepdim=True)

        all_embeddings.append(features.cpu().numpy())

    return np.vstack(all_embeddings)

def get_or_build_embeddings(df: pd.DataFrame, split_name: str):
    cache_path = CACHE_DIR / f"{split_name}_embeddings.npy"

    if CONFIG["cache_embeddings"] and cache_path.exists():
        print(f"[CACHE] Loading {split_name} embeddings from {cache_path}")
        return np.load(cache_path)

    emb = extract_clip_embeddings(
        image_paths=df["image_path"].tolist(),
        processor=PROCESSOR,
        model=CLIP,
        device=DEVICE,
        batch_size=CONFIG["batch_size"],
        normalize_embeddings=CONFIG["normalize_embeddings"],
    )

    if CONFIG["cache_embeddings"]:
        np.save(cache_path, emb)
        print(f"[CACHE] Saved {split_name} embeddings to {cache_path}")

    return emb

In [49]:
X_train = get_or_build_embeddings(train_df, "train")
X_test = get_or_build_embeddings(test_df, "test")
y = train_df["target"].values

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

Extracting CLIP embeddings: 100%|██████████| 150/150 [01:24<00:00,  1.78it/s]


[CACHE] Saved train embeddings to /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1/cache/train_embeddings.npy


Extracting CLIP embeddings: 100%|██████████| 65/65 [00:33<00:00,  1.97it/s]

[CACHE] Saved test embeddings to /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1/cache/test_embeddings.npy
X_train shape: (4800, 512)
X_test shape : (2058, 512)


## 4. Classifier utilities

In [50]:
def build_classifier():
    if CONFIG["classifier_type"] != "logreg":
        raise ValueError(f"Unsupported classifier_type: {CONFIG['classifier_type']}")

    class_weight = "balanced" if CONFIG["use_class_weights"] else None

    clf = LogisticRegression(
        C=CONFIG["logreg_C"],
        max_iter=CONFIG["logreg_max_iter"],
        class_weight=class_weight,
        random_state=CONFIG["random_state"],
    )
    return clf

## 5. One-fold training helper

In [51]:
def run_one_fold(fold, train_idx, val_idx, X, y):
    X_tr, X_va = X[train_idx], X[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    clf = build_classifier()
    clf.fit(X_tr, y_tr)

    val_probs = clf.predict_proba(X_va)[:, 1]
    val_preds = (val_probs >= 0.5).astype(int)
    val_f1 = f1_score(y_va, val_preds)

    fold_model_path = MODELS_DIR / f"fold_{fold}_classifier.joblib"
    try:
        import joblib
        joblib.dump(clf, fold_model_path)
    except Exception as e:
        print(f"[WARN] Could not save classifier for fold {fold}: {e}")

    fold_result = {
        "fold": fold,
        "val_idx": val_idx,
        "val_probs": val_probs,
        "val_f1_05": val_f1,
        "classifier": clf,
        "model_path": str(fold_model_path),
    }

    return fold_result

## 6. K-Fold training

In [52]:
skf = StratifiedKFold(
    n_splits=CONFIG["n_splits"],
    shuffle=True,
    random_state=CONFIG["random_state"],
)

oof_probs = np.zeros(len(train_df), dtype=np.float32)
fold_scores = []
fold_classifiers = []

best_fold = None
best_fold_f1 = -np.inf
best_fold_classifier = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y), start=1):
    result = run_one_fold(fold, train_idx, val_idx, X_train, y)

    oof_probs[result["val_idx"]] = result["val_probs"]
    fold_scores.append({
        "fold": result["fold"],
        "val_f1_05": result["val_f1_05"],
        "model_path": result["model_path"],
    })
    fold_classifiers.append(result["classifier"])

    print(f"Fold {fold} F1 @ 0.5: {result['val_f1_05']:.4f}")

    if result["val_f1_05"] > best_fold_f1:
        best_fold_f1 = result["val_f1_05"]
        best_fold = result["fold"]
        best_fold_classifier = result["classifier"]

fold_scores_df = pd.DataFrame(fold_scores)
display(fold_scores_df)

print("Best fold      :", best_fold)
print("Best fold F1   :", best_fold_f1)
print("Mean fold F1   :", fold_scores_df["val_f1_05"].mean())
print("Std fold F1    :", fold_scores_df["val_f1_05"].std(ddof=0))

fold_scores_df.to_csv(REPORTS_DIR / "fold_scores.csv", index=False)

Fold 1 F1 @ 0.5: 0.8191
Fold 2 F1 @ 0.5: 0.8480
Fold 3 F1 @ 0.5: 0.8223
Fold 4 F1 @ 0.5: 0.8244
Fold 5 F1 @ 0.5: 0.8109


,fold,val_f1_05,model_path
0,1,0.819127,/home/tbeliere/Projects/DCU/AML_PJ/outputs/exp...
1,2,0.848034,/home/tbeliere/Projects/DCU/AML_PJ/outputs/exp...
2,3,0.822270,/home/tbeliere/Projects/DCU/AML_PJ/outputs/exp...
3,4,0.824395,/home/tbeliere/Projects/DCU/AML_PJ/outputs/exp...
4,5,0.810870,/home/tbeliere/Projects/DCU/AML_PJ/outputs/exp...


Best fold      : 2
Best fold F1   : 0.8480340063761955
Mean fold F1   : 0.8249391142584385
Std fold F1    : 0.012430043288967949


## 7. OOF threshold tuning and evaluation

In [54]:
thresholds = np.linspace(0.1, 0.9, 81)

threshold_rows = []
best_threshold = 0.5
best_f1 = -np.inf

for thr in thresholds:
    preds = (oof_probs >= thr).astype(int)
    score = f1_score(y, preds)
    precision_ai = precision_score(y, preds, pos_label=1, zero_division=0)
    recall_ai = recall_score(y, preds, pos_label=1, zero_division=0)

    threshold_rows.append({
        "threshold": float(thr),
        "f1": float(score),
        "precision_ai_generated": float(precision_ai),
        "recall_ai_generated": float(recall_ai),
    })

    if score > best_f1:
        best_f1 = score
        best_threshold = float(thr)

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(REPORTS_DIR / "threshold_sweep.csv", index=False)

oof_pred_target_05 = (oof_probs >= 0.5).astype(int)
oof_pred_target_best = (oof_probs >= best_threshold).astype(int)

f1_05 = f1_score(y, oof_pred_target_05)
precision_ai = precision_score(y, oof_pred_target_best, pos_label=1, zero_division=0)
recall_ai = recall_score(y, oof_pred_target_best, pos_label=1, zero_division=0)

print("OOF F1 @ 0.5     :", f1_05)
print("Best threshold   :", best_threshold)
print("Best OOF F1      :", best_f1)
print("Precision (AI)   :", precision_ai)
print("Recall (AI)      :", recall_ai)

print("\nClassification report @ best threshold:")
print(classification_report(y, oof_pred_target_best, target_names=[id2label[0], id2label[1]], zero_division=0))

cm = confusion_matrix(y, oof_pred_target_best)
cm_df = pd.DataFrame(cm, index=[f"true_{id2label[0]}", f"true_{id2label[1]}"], columns=[f"pred_{id2label[0]}", f"pred_{id2label[1]}"])
display(cm_df)
cm_df.to_csv(REPORTS_DIR / "confusion_matrix_best_threshold.csv")

OOF F1 @ 0.5     : 0.8249787595581988
Best threshold   : 0.48
Best OOF F1      : 0.8300395256916996
Precision (AI)   : 0.800561797752809
Recall (AI)      : 0.8617710583153347

Classification report @ best threshold:
              precision    recall  f1-score   support

   authentic       0.86      0.80      0.83      2485
ai_generated       0.80      0.86      0.83      2315

    accuracy                           0.83      4800
   macro avg       0.83      0.83      0.83      4800
weighted avg       0.83      0.83      0.83      4800



,pred_authentic,pred_ai_generated
true_authentic,1988,497
true_ai_generated,320,1995


In [55]:
oof_df = pd.DataFrame({
    "image_id": train_df["image_id"],
    "label": train_df["label"],
    "target": train_df["target"],
    "oof_ai_generated_proba": oof_probs,
})

oof_df["oof_pred_target"] = (oof_df["oof_ai_generated_proba"] >= best_threshold).astype(int)
oof_df["oof_pred_label"] = oof_df["oof_pred_target"].map(id2label)

if CONFIG["save_oof_csv"]:
    oof_df.to_csv(REPORTS_DIR / "oof_predictions.csv", index=False)

display(oof_df.head())

,image_id,label,target,oof_ai_generated_proba,oof_pred_target,oof_pred_label
0,000541e8-6ea3-4ae0-a79b-10bd3bc3a38d.jpg,ai_generated,1,0.824660,1,ai_generated
1,00329fac-20e5-41f7-92df-05b256fdf9e8.jpg,ai_generated,1,0.612406,1,ai_generated
2,005af84d-d9ae-4842-a4a2-75fc8f590a94.jpg,ai_generated,1,0.553369,1,ai_generated
3,0068b91f-794a-4520-83a4-0f8735d2a229.jpg,ai_generated,1,0.676117,1,ai_generated
4,009ad604-af12-4c5d-b6a6-81da359c13a1.jpg,ai_generated,1,0.826246,1,ai_generated


## 8. Test prediction helpers

In [56]:
def predict_test_proba(classifier, X_test):
    return classifier.predict_proba(X_test)[:, 1]

## 9. Best-fold submission

In [59]:
best_fold_probs = predict_test_proba(best_fold_classifier, X_test)

best_fold_test_pred_df = pd.DataFrame({
    "image_id": test_df["image_id"],
    "ai_generated_proba": best_fold_probs,
    "pred_target": (best_fold_probs >= best_threshold).astype(int),
})
best_fold_test_pred_df["pred_label"] = best_fold_test_pred_df["pred_target"].map(id2label)

if CONFIG["save_test_probs_csv"]:
    best_fold_test_pred_df.to_csv(REPORTS_DIR / "test_probabilities_best_fold.csv", index=False)

submission_best_fold = pd.DataFrame({
    "image_id": test_df["image_id"],
    "ground_truth": best_fold_test_pred_df["pred_target"],
})

if CONFIG["save_submission_csv"]:
    submission_best_fold.to_csv(REPORTS_DIR / "submission_best_fold.csv", index=False)

display(best_fold_test_pred_df.head())

,image_id,ai_generated_proba,pred_target,pred_label
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,0.159069,0,authentic
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,0.612158,1,ai_generated
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,0.092182,0,authentic
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,0.554366,1,ai_generated
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,0.097420,0,authentic


## 10. Fold-ensemble submission

In [60]:
fold_test_probs = []

for clf in fold_classifiers:
    fold_test_probs.append(predict_test_proba(clf, X_test))

fold_test_probs = np.vstack(fold_test_probs)
ensemble_test_probs = fold_test_probs.mean(axis=0)

fold_ensemble_test_pred_df = pd.DataFrame({
    "image_id": test_df["image_id"],
    "ai_generated_proba": ensemble_test_probs,
    "pred_target": (ensemble_test_probs >= best_threshold).astype(int),
})
fold_ensemble_test_pred_df["pred_label"] = fold_ensemble_test_pred_df["pred_target"].map(id2label)

if CONFIG["save_test_probs_csv"]:
    fold_ensemble_test_pred_df.to_csv(REPORTS_DIR / "test_probabilities.csv", index=False)

submission_fold_ensemble = pd.DataFrame({
    "image_id": test_df["image_id"],
    "ground_truth": fold_ensemble_test_pred_df["pred_target"],
})

if CONFIG["save_submission_csv"]:
    submission_fold_ensemble.to_csv(REPORTS_DIR / "submission_fold_ensemble.csv", index=False)

display(fold_ensemble_test_pred_df.head())

,image_id,ai_generated_proba,pred_target,pred_label
0,000317ef-6928-48ca-9b51-7742ab8dfb86.jpg,0.167540,0,authentic
1,000a811d-cf8f-4b5b-be75-bd9f1a73c098.jpg,0.614467,1,ai_generated
2,003a1394-83c5-41c0-9cd5-18ac89397c7b.jpg,0.086512,0,authentic
3,00888016-16c4-4ce7-a03e-073f04a1a9c8.jpg,0.605248,1,ai_generated
4,00e301e1-7520-4abb-8007-9e00a0b04da6.jpg,0.104191,0,authentic


## 11. Metrics summary

In [62]:
metrics = {
    "f1_05": float(f1_05),
    "best_threshold": float(best_threshold),
    "best_f1": float(best_f1),
    "precision_ai_generated": float(precision_ai),
    "recall_ai_generated": float(recall_ai),
    "fold_f1_mean_05": float(fold_scores_df["val_f1_05"].mean()),
    "fold_f1_std_05": float(fold_scores_df["val_f1_05"].std(ddof=0)),
    "best_fold": int(best_fold),
    "best_fold_f1_05": float(best_fold_f1),
    "train_manifest": str(TRAIN_MANIFEST_PATH),
    "test_manifest": str(TEST_MANIFEST_PATH),
    "clip_model_name": CONFIG["clip_model_name"],
    "classifier_type": CONFIG["classifier_type"],
    "use_class_weights": bool(CONFIG["use_class_weights"]),
}

with open(REPORTS_DIR / "metrics_summary.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(json.dumps(metrics, indent=4))
print(f"Saved outputs to: {REPORTS_DIR}")

{
    "f1_05": 0.8249787595581988,
    "best_threshold": 0.48,
    "best_f1": 0.8300395256916996,
    "precision_ai_generated": 0.800561797752809,
    "recall_ai_generated": 0.8617710583153347,
    "fold_f1_mean_05": 0.8249391142584385,
    "fold_f1_std_05": 0.012430043288967949,
    "best_fold": 2,
    "best_fold_f1_05": 0.8480340063761955,
    "train_manifest": "/home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/train_manifest_etsy_only.csv",
    "test_manifest": "/home/tbeliere/Projects/DCU/AML_PJ/outputs/manifests/test_manifest_etsy.csv",
    "clip_model_name": "openai/clip-vit-base-patch32",
    "classifier_type": "logreg",
    "use_class_weights": true
}
Saved outputs to: /home/tbeliere/Projects/DCU/AML_PJ/outputs/experiments/clip_logreg_etsy_only_v1/reports


## 12. Notes

- Compare experiments using the saved `oof_predictions.csv` probabilities, not only the hard predictions.
- For later ensemble work, `submission_fold_ensemble.csv` is usually the stronger file to keep.
- If you switch to `train_manifest_etsy_plus_external.csv`, start by changing:
  - `train_manifest_relpath`
  - `experiment_name`
  - optionally `use_class_weights=True`